# 02 — GitHub verisini okuma ve 2D Mordred feature store oluşturma

Bu sürümde GitHub bağlantısı tek bir sabit dosya yoluna bağlı değildir.

Notebook önce hedefe özel Lipinski filtreli CSV dosyasını arar. Dosya `data/` klasöründe değilse repo kökünü ve alt klasörleri kontrol eder. Hedef dosya henüz GitHub'a yüklenmemişse mevcut repodaki `training_data.csv` dosyasını fallback olarak kullanır.

Yalnızca **2D Mordred descriptorları** üretilir. Bütün çıktılar Google Drive içindeki ortak workshop klasörüne kaydedilir.

## Workshop veri akışı

```text
GitHub repo dosya ağacı
        ↓
Uygun regression CSV'nin otomatik bulunması
        ↓
Target ve SMILES kontrolü
        ↓
Canonical SMILES ve duplicate temizliği
        ↓
Mordred 2D descriptor üretimi
        ↓
Eksik, sabit ve yüksek korelasyonlu feature temizliği
        ↓
Google Drive kayıtları
```

GitHub deposu:

```text
https://github.com/MOL-FEST/MOL-FET_regression/tree/main
```

Google Drive çıktı klasörü:

```text
MyDrive/MOL_FET_regression_workshop/
```

## Hücre 1 — Paket kontrolü

In [ ]:
import importlib.util
# Paketlerin aktif Python ortamında kurulu olup olmadığını kontrol etmek için kullanılır.

import subprocess
# Eksik paketlerin pip ile kurulmasını sağlamak için kullanılır.

import sys
# Colab oturumunda kullanılan aktif Python yorumlayıcısının yolunu verir.


BASE_PACKAGES = [
    ("numpy", "numpy"),
    # Sayısal matris işlemleri için kullanılır.

    ("pandas", "pandas"),
    # CSV okuma, temizleme ve çıktı kaydı için kullanılır.

    ("requests", "requests"),
    # GitHub API ve RAW bağlantılarına erişmek için kullanılır.

    ("matplotlib", "matplotlib"),
    # Target dağılım grafiğini çizmek için kullanılır.

    ("sklearn", "scikit-learn"),
    # Eksik değer doldurma ve feature ön işleme için kullanılır.

    ("rdkit", "rdkit"),
    # SMILES doğrulama ve molekül nesnesi üretmek için kullanılır.
]
# Temel paketlerin import ve pip adları tanımlanır.


for import_name, pip_name in BASE_PACKAGES:
    # Paketler sırayla kontrol edilir.

    if importlib.util.find_spec(import_name) is None:
        # Paket bulunamazsa aktif Python ortamına kurulur.

        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                pip_name,
            ]
        )
        # Güncel paket pip üzerinden kurulur.


subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        "mordredcommunity[full]==2.0.7",
    ]
)
# Modern Python ve pandas sürümleriyle uyumlu Mordred Community paketi kurulur.


print("✅ CHECKPOINT 02.1: Paket kontrolü tamamlandı.")

## Hücre 2 — Kütüphanelerin içe aktarılması

In [ ]:
from pathlib import Path
# Google Drive ve çıktı dosya yollarını yönetmek için kullanılır.

from io import BytesIO
# GitHub'dan indirilen CSV içeriğini bellekte pandas'a aktarmak için kullanılır.

import json
# Feature manifest dosyasını JSON olarak kaydetmek için kullanılır.

import warnings
# Gereksiz uyarıları azaltmak için kullanılır.

warnings.filterwarnings("ignore")
# Workshop ekranını kalabalıklaştıran genel uyarılar gizlenir.

import numpy as np
# Sayısal descriptor, korelasyon ve sonsuz değer işlemleri için kullanılır.

import pandas as pd
# CSV okuma, veri temizleme ve feature store oluşturmak için kullanılır.

import requests
# GitHub API ve RAW bağlantılarına istek göndermek için kullanılır.

import matplotlib.pyplot as plt
# Target dağılım grafiğini çizmek için kullanılır.

from IPython.display import display
# DataFrame tablolarını notebook içinde göstermek için kullanılır.

from google.colab import drive
# Çıktıları Google Drive'a kaydetmek için kullanılır.

from rdkit import Chem
# SMILES doğrulama ve molekül nesnesi oluşturmak için kullanılır.

from mordred import Calculator, descriptors
# Yalnızca 2D Mordred descriptorlarını hesaplamak için kullanılır.

from sklearn.impute import SimpleImputer
# Eksik descriptor değerlerini medyanla doldurmak için kullanılır.


print("RDKit sürümü:", Chem.rdBase.rdkitVersion)
# Kullanılan RDKit sürümü kontrol amacıyla yazdırılır.

print("Pandas sürümü:", pd.__version__)
# Kullanılan pandas sürümü kontrol amacıyla yazdırılır.

print("✅ CHECKPOINT 02.2: Kütüphaneler başarıyla içe aktarıldı.")

## Hücre 3 — Google Drive bağlantısı

In [ ]:
drive.mount(
    "/content/drive",
    force_remount=False,
)
# Google Drive standart Colab dizinine bağlanır.


print("✅ CHECKPOINT 02.3: Google Drive bağlantısı hazır.")

## Hücre 4 — Target, GitHub ve feature ayarları

In [ ]:
TARGET_ID = "CHEMBL206"
# İşlenecek ChEMBL Target ID burada değiştirilir.


GITHUB_OWNER = "MOL-FEST"
# GitHub organizasyon adı tanımlanır.


GITHUB_REPOSITORY = "MOL-FET_regression"
# GitHub repo adı tanımlanır.


GITHUB_BRANCH = "main"
# Dosyaların okunacağı branch tanımlanır.


PREFERRED_INPUT_FILENAME = (
    f"{TARGET_ID}_IC50_parent_dedup_Lipinski_filtered.csv"
)
# Tercih edilen Lipinski filtreli girdi dosya adı oluşturulur.


RAW_INPUT_FILENAME = (
    f"{TARGET_ID}_IC50_single_protein_format_CLEAN_parent_dedup.csv"
)
# Filtreli dosya henüz yüklenmemişse kullanılabilecek ham parent-dedup dosya adı oluşturulur.


DRIVE_ROOT = Path(
    "/content/drive/MyDrive/MOL_FET_regression_workshop"
)
# Bütün feature çıktılarının kaydedileceği ortak Drive klasörü tanımlanır.


DRIVE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)
# Drive klasörü yoksa oluşturulur.


MAX_MISSING_RATIO = 0.20
# Bir descriptorın izin verilen maksimum eksik değer oranı tanımlanır.


CORRELATION_THRESHOLD = 0.95
# Yüksek korelasyonlu descriptorları çıkarmak için eşik tanımlanır.


MORDRED_NPROC = 1
# Workshop ortamında kararlı bellek kullanımı için tek işlem kullanılır.


REQUEST_TIMEOUT = 180
# GitHub isteklerinin zaman aşımı süresi saniye olarak belirlenir.


print("TARGET_ID:", TARGET_ID)
print("Drive çıktı klasörü:", DRIVE_ROOT)
print("✅ CHECKPOINT 02.4: Target, GitHub ve feature ayarları hazır.")

## Hücre 5 — Aday GitHub dosya yolları

In [ ]:
PREFERRED_GITHUB_PATHS = [
    f"data/{PREFERRED_INPUT_FILENAME}",
    # Tercih edilen data/ klasörü yolu.

    PREFERRED_INPUT_FILENAME,
    # Filtreli CSV repo köküne yüklenmişse kullanılacak yol.

    "training_data.csv",
    # Mevcut MOL-FET regression reposundaki eğitim verisi fallback yoludur.

    "data/training_data.csv",
    # training_data.csv data/ klasörüne taşınırsa kullanılacak yol.

    f"data/{RAW_INPUT_FILENAME}",
    # Ham parent-dedup CSV data/ klasöründe bulunursa kullanılacak yol.

    RAW_INPUT_FILENAME,
    # Ham parent-dedup CSV repo kökünde bulunursa kullanılacak yol.
]
# GitHub içinde denenecek dosya yolları öncelik sırasıyla tanımlanır.


GITHUB_API_TREE_URL = (
    f"https://api.github.com/repos/{GITHUB_OWNER}/"
    f"{GITHUB_REPOSITORY}/git/trees/{GITHUB_BRANCH}?recursive=1"
)
# Repo içindeki bütün dosyaları recursive olarak listeleyen GitHub API adresi oluşturulur.


print("GitHub dosya adayları:")
for candidate in PREFERRED_GITHUB_PATHS:
    print(" -", candidate)


print("✅ CHECKPOINT 02.5: GitHub aday yolları tanımlandı.")

## Hücre 6 — RAW GitHub URL oluşturma fonksiyonu

In [ ]:
def build_raw_url(relative_path):
    """Repo içindeki göreli dosya yolunu GitHub RAW URL adresine dönüştürür."""

    clean_path = str(relative_path).lstrip("/")
    # Yolun başındaki olası eğik çizgi kaldırılır.

    raw_url = (
        f"https://raw.githubusercontent.com/"
        f"{GITHUB_OWNER}/{GITHUB_REPOSITORY}/"
        f"{GITHUB_BRANCH}/{clean_path}"
    )
    # Tam RAW GitHub adresi oluşturulur.

    return raw_url
    # RAW URL döndürülür.


print("✅ CHECKPOINT 02.6: RAW URL fonksiyonu hazır.")

## Hücre 7 — GitHub repo ağacını okuma fonksiyonu

In [ ]:
def get_repository_file_paths():
    """GitHub API üzerinden repo içindeki bütün dosya yollarını döndürür."""

    response = requests.get(
        GITHUB_API_TREE_URL,
        timeout=REQUEST_TIMEOUT,
        headers={
            "Accept": "application/vnd.github+json",
        },
    )
    # GitHub API üzerinden recursive repo ağacı istenir.

    response.raise_for_status()
    # HTTP hata kodu varsa açıklayıcı exception üretilir.

    payload = response.json()
    # JSON yanıtı Python sözlüğüne dönüştürülür.

    tree_entries = payload.get("tree", [])
    # Repo ağacındaki bütün girişler alınır.

    file_paths = [
        entry.get("path")
        for entry in tree_entries
        if entry.get("type") == "blob" and entry.get("path")
    ]
    # Yalnızca gerçek dosya girişlerinin yolları seçilir.

    if not file_paths:
        # API boş dosya listesi döndürürse kontrol edilir.

        raise RuntimeError(
            "GitHub API repo ağacında dosya döndürmedi."
        )
        # Boş repo ağacıyla devam edilmez.

    return file_paths
    # Bütün dosya yolları döndürülür.


print("✅ CHECKPOINT 02.7: GitHub repo ağacı fonksiyonu hazır.")

## Hücre 8 — Uygun CSV yolunu otomatik seçme fonksiyonu

In [ ]:
def resolve_github_csv_path():
    """Repo içinde kullanılabilir regression CSV yolunu otomatik seçer."""

    try:
        # Öncelikle GitHub API üzerinden repo dosya ağacı okunur.

        repository_files = get_repository_file_paths()
        # Repo içindeki bütün dosya yolları alınır.

        repository_file_set = set(repository_files)
        # Tam yol kontrollerini hızlandırmak için küme oluşturulur.

        for preferred_path in PREFERRED_GITHUB_PATHS:
            # Öncelikli yollar sırayla kontrol edilir.

            if preferred_path in repository_file_set:
                # Aday yol repoda gerçekten varsa seçilir.

                return preferred_path
                # İlk bulunan uygun yol döndürülür.

        preferred_names = {
            PREFERRED_INPUT_FILENAME,
            RAW_INPUT_FILENAME,
            "training_data.csv",
        }
        # Alt klasör konumu değişmiş olsa bile aranacak dosya adları tanımlanır.

        recursive_matches = [
            path
            for path in repository_files
            if Path(path).name in preferred_names
        ]
        # Dosya adı eşleşen bütün recursive sonuçlar belirlenir.

        if recursive_matches:
            # En az bir eşleşme bulunduysa kontrol edilir.

            recursive_matches = sorted(
                recursive_matches,
                key=lambda path: (
                    Path(path).name != PREFERRED_INPUT_FILENAME,
                    Path(path).name != "training_data.csv",
                    len(Path(path).parts),
                    path,
                ),
            )
            # Filtreli target dosyası, ardından training_data ve daha kısa yollar önceliklendirilir.

            return recursive_matches[0]
            # En uygun eşleşme döndürülür.

        csv_files = sorted(
            path
            for path in repository_files
            if str(path).lower().endswith(".csv")
        )
        # Tanı koymak için repodaki bütün CSV dosyaları belirlenir.

        raise FileNotFoundError(
            "Uygun regression CSV bulunamadı.\n"
            "Repo içindeki CSV dosyaları:\n- "
            + "\n- ".join(csv_files[:50])
        )
        # Hiç uygun dosya bulunamazsa mevcut CSV yolları hata mesajına eklenir.

    except requests.RequestException as api_error:
        # GitHub API geçici olarak çalışmazsa hata yakalanır.

        print("GitHub API erişimi başarısız:", api_error)
        # API hatası bilgi amaçlı gösterilir.

        return None
        # Doğrudan RAW adaylarını denemek üzere None döndürülür.


print("✅ CHECKPOINT 02.8: GitHub CSV çözümleme fonksiyonu hazır.")

## Hücre 9 — CSV sütun uygunluğunu kontrol etme fonksiyonu

In [ ]:
TARGET_COLUMN_CANDIDATES = [
    "pStandard",
    "pStandard_mean",
    "pIC50",
    "target",
    "y",
]
# Olası regression target sütun adları tanımlanır.


SMILES_COLUMN_CANDIDATES = [
    "parent_smiles",
    "canonical_smiles",
    "smiles",
    "SMILES",
]
# Olası SMILES sütun adları tanımlanır.


ID_COLUMN_CANDIDATES = [
    "parent_chembl_id",
    "molecule_chembl_id",
    "chembl_id",
    "ID",
    "id",
]
# Olası molekül kimliği sütun adları tanımlanır.


def detect_column(dataframe, candidates, label, required=True):
    """Aday sütun adları arasından DataFrame içinde bulunan ilk sütunu döndürür."""

    for candidate in candidates:
        # Aday sütunlar verilen sırayla kontrol edilir.

        if candidate in dataframe.columns:
            # Sütun DataFrame içinde bulunursa seçilir.

            return candidate
            # İlk bulunan uygun sütun döndürülür.

    if required:
        # Sütun zorunluysa kontrol edilir.

        raise ValueError(
            f"{label} sütunu bulunamadı. Adaylar: {candidates}"
        )
        # Zorunlu sütun bulunamazsa hata oluşturulur.

    return None
    # Zorunlu olmayan sütun bulunamazsa None döndürülür.


def dataframe_has_required_columns(dataframe):
    """CSV'nin en az bir target ve bir SMILES sütunu içerip içermediğini kontrol eder."""

    target_found = any(
        column in dataframe.columns
        for column in TARGET_COLUMN_CANDIDATES
    )
    # En az bir target sütunu olup olmadığı kontrol edilir.

    smiles_found = any(
        column in dataframe.columns
        for column in SMILES_COLUMN_CANDIDATES
    )
    # En az bir SMILES sütunu olup olmadığı kontrol edilir.

    return target_found and smiles_found
    # Her iki kritik sütun grubu mevcutsa True döndürülür.


print("✅ CHECKPOINT 02.9: Sütun doğrulama fonksiyonları hazır.")

## Hücre 10 — GitHub CSV indirme ve doğrulama fonksiyonu

In [ ]:
def download_and_validate_csv(relative_path):
    """Bir GitHub yolundaki CSV'yi indirir ve regression yapısını doğrular."""

    raw_url = build_raw_url(relative_path)
    # Göreli yol tam RAW URL adresine dönüştürülür.

    print("Denenen GitHub RAW URL:", raw_url)
    # Test edilen bağlantı ekrana yazdırılır.

    response = requests.get(
        raw_url,
        timeout=REQUEST_TIMEOUT,
    )
    # CSV içeriği GitHub RAW bağlantısından istenir.

    if response.status_code == 404:
        # Dosya belirtilen konumda yoksa kontrol edilir.

        print("  → 404: Bu konumda dosya yok.")
        # 404 bilgisi yazdırılır.

        return None, raw_url
        # Pipeline durmadan sonraki adaya geçilir.

    response.raise_for_status()
    # 404 dışındaki HTTP hatalarında exception oluşturulur.

    if not response.content:
        # İndirilen içerik boşsa kontrol edilir.

        print("  → Dosya boş.")
        # Boş dosya bilgisi yazdırılır.

        return None, raw_url
        # Boş dosya kullanılmaz.

    try:
        # CSV ayrıştırma hatasını yakalamak için try bloğu kullanılır.

        dataframe = pd.read_csv(
            BytesIO(response.content),
            low_memory=False,
        )
        # İndirilen içerik pandas DataFrame'e dönüştürülür.

    except Exception as csv_error:
        # CSV olarak okunamayan içerik yakalanır.

        print("  → CSV okuma hatası:", csv_error)
        # Ayrıştırma hatası gösterilir.

        return None, raw_url
        # Geçersiz dosya kullanılmaz.

    if dataframe.empty:
        # DataFrame boşsa kontrol edilir.

        print("  → CSV boş.")
        # Boş tablo bilgisi gösterilir.

        return None, raw_url
        # Boş DataFrame kullanılmaz.

    if not dataframe_has_required_columns(dataframe):
        # Target ve SMILES sütunları kontrol edilir.

        print(
            "  → Gerekli target/SMILES sütunları yok:",
            list(dataframe.columns),
        )
        # Uygun olmayan sütun yapısı gösterilir.

        return None, raw_url
        # Modelleme yapısına uymayan CSV atlanır.

    print("  → Uygun regression CSV bulundu.")
    # Başarılı aday bilgisi gösterilir.

    return dataframe, raw_url
    # Geçerli DataFrame ve kullanılan URL döndürülür.


print("✅ CHECKPOINT 02.10: CSV indirme fonksiyonu hazır.")

## Hücre 11 — GitHub'daki uygun regression CSV'nin okunması

In [ ]:
resolved_path = resolve_github_csv_path()
# GitHub API ile en uygun CSV yolu bulunmaya çalışılır.


paths_to_try = []
# Denenecek GitHub yollarını tutacak liste oluşturulur.


if resolved_path is not None:
    # API bir dosya yolu bulduysa kontrol edilir.

    paths_to_try.append(resolved_path)
    # Bulunan yol ilk sıraya eklenir.


for preferred_path in PREFERRED_GITHUB_PATHS:
    # Öncelikli yollar sırayla dolaşılır.

    if preferred_path not in paths_to_try:
        # Yol daha önce eklenmediyse kontrol edilir.

        paths_to_try.append(preferred_path)
        # Yol deneme listesine eklenir.


df_raw = None
# Başlangıçta henüz veri okunmadığı belirtilir.


USED_GITHUB_URL = None
# Kullanılan başarılı RAW URL daha sonra manifestte saklanmak üzere boş tanımlanır.


USED_GITHUB_PATH = None
# Kullanılan repo içi yol daha sonra raporlanmak üzere boş tanımlanır.


for candidate_path in paths_to_try:
    # Bütün aday yollar sırayla denenir.

    candidate_df, candidate_url = download_and_validate_csv(
        candidate_path
    )
    # Aday CSV indirilir ve doğrulanır.

    if candidate_df is not None:
        # Geçerli DataFrame bulunduysa kontrol edilir.

        df_raw = candidate_df
        # Geçerli veri ana DataFrame olarak atanır.

        USED_GITHUB_URL = candidate_url
        # Başarılı RAW URL saklanır.

        USED_GITHUB_PATH = candidate_path
        # Başarılı repo yolu saklanır.

        break
        # İlk geçerli dosya bulunduğunda döngü sonlandırılır.


if df_raw is None:
    # Hiçbir adaydan geçerli veri okunamadıysa kontrol edilir.

    raise FileNotFoundError(
        "GitHub reposunda target ve SMILES sütunlarını içeren "
        "uygun regression CSV bulunamadı."
    )
    # Açıklayıcı hata oluşturulur.


print("Kullanılan GitHub yolu:", USED_GITHUB_PATH)
print("Kullanılan GitHub RAW URL:", USED_GITHUB_URL)
print("Ham veri boyutu:", df_raw.shape)
display(df_raw.head(10))
print("✅ CHECKPOINT 02.11: GitHub regression CSV başarıyla okundu.")

## Hücre 12 — Canonical SMILES fonksiyonu

In [ ]:
def canonicalize_smiles(smiles):
    """SMILES metnini doğrular ve canonical isomeric SMILES döndürür."""

    if pd.isna(smiles):
        # Eksik SMILES kontrol edilir.

        return None
        # Eksik değer geçersiz kabul edilir.

    text = str(smiles).strip()
    # SMILES temiz metne dönüştürülür.

    if not text:
        # Boş metin kontrol edilir.

        return None
        # Boş SMILES geçersiz kabul edilir.

    try:
        # RDKit ayrıştırma hatalarını yakalamak için try bloğu kullanılır.

        molecule = Chem.MolFromSmiles(text)
        # SMILES RDKit molekül nesnesine dönüştürülür.

        if molecule is None:
            # Molekül oluşturulamazsa kontrol edilir.

            return None
            # Geçersiz SMILES için None döndürülür.

        Chem.SanitizeMol(molecule)
        # Molekülün kimyasal geçerliliği kontrol edilir.

        return Chem.MolToSmiles(
            molecule,
            canonical=True,
            isomericSmiles=True,
        )
        # Canonical ve stereokimyayı koruyan SMILES döndürülür.

    except Exception:
        # Beklenmeyen RDKit hataları yakalanır.

        return None
        # Hatalı yapı geçersiz kabul edilir.


print("✅ CHECKPOINT 02.12: Canonical SMILES fonksiyonu hazır.")

## Hücre 13 — Target, SMILES ve molekül ID sütunlarının seçilmesi

In [ ]:
target_column = detect_column(
    df_raw,
    TARGET_COLUMN_CANDIDATES,
    "Regression target",
)
# Regression target sütunu otomatik seçilir.


smiles_column = detect_column(
    df_raw,
    SMILES_COLUMN_CANDIDATES,
    "SMILES",
)
# SMILES sütunu otomatik seçilir.


id_column = detect_column(
    df_raw,
    ID_COLUMN_CANDIDATES,
    "Molekül ID",
    required=False,
)
# Molekül ID sütunu varsa otomatik seçilir.


print("Target sütunu:", target_column)
print("SMILES sütunu:", smiles_column)
print("ID sütunu:", id_column)
print("✅ CHECKPOINT 02.13: Kritik sütunlar belirlendi.")

## Hücre 14 — Target değerlerinin hazırlanması

In [ ]:
df_work = df_raw.copy()
# Ham veri korunarak çalışma kopyası oluşturulur.


df_work["target"] = pd.to_numeric(
    df_work[target_column],
    errors="coerce",
)
# Seçilen regression target sütunu sayısal target sütununa dönüştürülür.


print("Eksik/geçersiz target:", int(df_work["target"].isna().sum()))
print("✅ CHECKPOINT 02.14: Target değerleri hazırlandı.")

## Hücre 15 — SMILES değerlerinin canonical forma dönüştürülmesi

In [ ]:
df_work["canonical_smiles"] = (
    df_work[smiles_column]
    .apply(canonicalize_smiles)
)
# Her SMILES doğrulanıp canonical isomeric SMILES'a dönüştürülür.


print(
    "Geçersiz canonical SMILES:",
    int(df_work["canonical_smiles"].isna().sum()),
)
print("✅ CHECKPOINT 02.15: Canonical SMILES değerleri hazırlandı.")

## Hücre 16 — Molekül kimliklerinin hazırlanması

In [ ]:
if id_column is not None:
    # Molekül ID sütunu bulunmuşsa kontrol edilir.

    df_work["molecule_id"] = (
        df_work[id_column].astype(str)
    )
    # Var olan molekül kimlikleri standart molecule_id sütununa kopyalanır.

else:
    # Molekül ID sütunu yoksa alternatif kola geçilir.

    df_work["molecule_id"] = [
        f"ROW_{index:06d}"
        for index in range(len(df_work))
    ]
    # Her satır için tekrar üretilebilir geçici kimlik oluşturulur.


print("✅ CHECKPOINT 02.16: Molekül kimlikleri hazırlandı.")

## Hücre 17 — Geçersiz kayıtların ayrılması

In [ ]:
invalid_mask = (
    df_work["target"].isna()
    | df_work["canonical_smiles"].isna()
)
# Eksik target veya geçersiz SMILES içeren satırlar belirlenir.


invalid_rows = df_work.loc[
    invalid_mask
].copy()
# Geçersiz satırlar ayrı kalite raporuna alınır.


df_clean = df_work.loc[
    ~invalid_mask
].copy()
# Yalnızca geçerli target ve SMILES içeren satırlar tutulur.


if df_clean.empty:
    # Temizlik sonrasında molekül kalıp kalmadığı kontrol edilir.

    raise ValueError(
        "Target ve SMILES temizliğinden sonra molekül kalmadı."
    )
    # Boş veriyle descriptor üretimine geçilmez.


print("Geçerli satır:", len(df_clean))
print("Geçersiz satır:", len(invalid_rows))
print("✅ CHECKPOINT 02.17: Geçersiz kayıtlar ayrıldı.")

## Hücre 18 — Canonical duplicate kayıtların birleştirilmesi

In [ ]:
duplicate_summary = (
    df_clean
    .groupby(
        "canonical_smiles",
        as_index=False,
    )
    .agg(
        molecule_id=(
            "molecule_id",
            lambda values: ";".join(
                sorted(set(map(str, values)))
            ),
        ),
        target=(
            "target",
            "median",
        ),
        target_mean=(
            "target",
            "mean",
        ),
        target_std=(
            "target",
            "std",
        ),
        source_rows=(
            "target",
            "size",
        ),
    )
)
# Aynı canonical SMILES'a ait target kayıtları medyan değerle tek satırda birleştirilir.


duplicate_summary["target_std"] = (
    duplicate_summary["target_std"].fillna(0.0)
)
# Tek ölçümlü moleküllerdeki boş standart sapma sıfırla doldurulur.


df_clean = duplicate_summary.reset_index(drop=True)
# Birleştirilmiş tablo final temiz veri olarak atanır.


if not df_clean["canonical_smiles"].is_unique:
    # Duplicate sonrasında tekrar kalıp kalmadığı kontrol edilir.

    raise AssertionError(
        "Canonical duplicate kayıtlar tamamen birleştirilemedi."
    )
    # Tekrar kaldıysa pipeline durdurulur.


print("Feature üretilecek benzersiz molekül:", len(df_clean))
print("✅ CHECKPOINT 02.18: Canonical duplicate kayıtlar birleştirildi.")

## Hücre 19 — Target dağılımının çizilmesi

In [ ]:
target_plot_path = (
    DRIVE_ROOT
    / f"{TARGET_ID}_target_distribution.png"
)
# Target dağılım grafiği için Drive yolu oluşturulur.


plt.figure(
    figsize=(9, 6),
)
# Yeni grafik figürü oluşturulur.


plt.hist(
    df_clean["target"],
    bins=30,
    edgecolor="black",
)
# Target değerleri histogram olarak çizilir.


plt.xlabel(
    target_column,
)
# X eksenine kaynak target sütun adı yazılır.


plt.ylabel(
    "Molekül sayısı",
)
# Y eksenine molekül sayısı etiketi yazılır.


plt.title(
    f"{TARGET_ID} — Regression target dağılımı",
)
# Grafiğe target kimliğini içeren başlık eklenir.


plt.tight_layout()
# Grafik elemanlarının taşması engellenir.


plt.savefig(
    target_plot_path,
    dpi=300,
    bbox_inches="tight",
)
# Target dağılım grafiği Google Drive'a kaydedilir.


plt.show()
# Grafik notebook ekranında gösterilir.


print("✅ CHECKPOINT 02.19: Target dağılım grafiği kaydedildi.")

## Hücre 20 — Target istatistiklerinin kontrolü

In [ ]:
display(
    df_clean["target"]
    .describe()
    .to_frame("target")
)
# Regression target için tanımlayıcı istatistikler gösterilir.


print("✅ CHECKPOINT 02.20: Target dağılımı kontrol edildi.")

## Hücre 21 — RDKit molekül nesnelerinin oluşturulması

In [ ]:
molecules = [
    Chem.MolFromSmiles(smiles)
    for smiles in df_clean["canonical_smiles"]
]
# Canonical SMILES değerleri RDKit molekül nesnelerine dönüştürülür.


if any(
    molecule is None
    for molecule in molecules
):
    # Beklenmeyen geçersiz molekül kalıp kalmadığı kontrol edilir.

    raise ValueError(
        "Mordred öncesinde geçersiz RDKit molekülü bulundu."
    )
    # Geçersiz molekül varsa descriptor hesabı başlatılmaz.


print("RDKit molekül sayısı:", len(molecules))
print("✅ CHECKPOINT 02.21: RDKit molekül nesneleri hazırlandı.")

## Hücre 22 — Mordred 2D hesaplayıcısının oluşturulması

In [ ]:
calculator = Calculator(
    descriptors,
    ignore_3D=True,
)
# Yalnızca 2D descriptorları içeren Mordred hesaplayıcısı oluşturulur.


print(
    "Hesaplanacak ham descriptor sayısı:",
    len(calculator.descriptors),
)
# Hesaplanacak ham Mordred descriptor sayısı yazdırılır.


print("✅ CHECKPOINT 02.22: Mordred 2D hesaplayıcısı hazır.")

## Hücre 23 — Ham Mordred 2D descriptorlarının hesaplanması

In [ ]:
mordred_raw = calculator.pandas(
    molecules,
    nproc=MORDRED_NPROC,
    quiet=False,
)
# Bütün moleküller için Mordred 2D descriptorları hesaplanır.


mordred_raw.columns = [
    str(column)
    for column in mordred_raw.columns
]
# Descriptor sütun adları güvenli metne dönüştürülür.


if mordred_raw.shape[0] != len(df_clean):
    # Descriptor satır sayısının molekül sayısıyla eşleşmesi kontrol edilir.

    raise AssertionError(
        "Mordred satır sayısı molekül sayısıyla uyuşmuyor."
    )
    # Satır sayısı uyuşmazsa pipeline durdurulur.


print("Ham Mordred matrisi:", mordred_raw.shape)
print("✅ CHECKPOINT 02.23: Ham Mordred descriptorları üretildi.")

## Hücre 24 — Mordred değerlerinin sayısala dönüştürülmesi

In [ ]:
mordred_numeric = mordred_raw.apply(
    pd.to_numeric,
    errors="coerce",
)
# Mordred hata nesneleri ve metin değerleri sayısala dönüştürülür.


mordred_numeric = mordred_numeric.replace(
    [
        np.inf,
        -np.inf,
    ],
    np.nan,
)
# Pozitif ve negatif sonsuz değerler eksik değer olarak işaretlenir.


print("Sayısal Mordred matrisi:", mordred_numeric.shape)
print("✅ CHECKPOINT 02.24: Mordred değerleri sayısala dönüştürüldü.")

## Hücre 25 — Yüksek eksik oranlı descriptorların çıkarılması

In [ ]:
missing_ratio = mordred_numeric.isna().mean()
# Her descriptor için eksik değer oranı hesaplanır.


high_missing_columns = missing_ratio[
    missing_ratio > MAX_MISSING_RATIO
].index.tolist()
# Eksik oranı belirlenen eşiği aşan descriptorlar seçilir.


X_step1 = mordred_numeric.drop(
    columns=high_missing_columns,
).copy()
# Yüksek eksiklik üreten descriptorlar matristen çıkarılır.


if X_step1.empty:
    # Filtre sonrasında descriptor kalıp kalmadığı kontrol edilir.

    raise ValueError(
        "Eksik değer filtresinden sonra descriptor kalmadı."
    )
    # Boş feature matrisiyle devam edilmez.


print("Yüksek eksik nedeniyle çıkarılan:", len(high_missing_columns))
print("Kalan descriptor:", X_step1.shape[1])
print("✅ CHECKPOINT 02.25: Eksik oran filtresi tamamlandı.")

## Hücre 26 — Kalan eksik değerlerin medyanla doldurulması

In [ ]:
imputer = SimpleImputer(
    strategy="median",
)
# Eksik descriptor değerlerini medyanla dolduracak imputer oluşturulur.


X_imputed = pd.DataFrame(
    imputer.fit_transform(X_step1),
    columns=X_step1.columns,
    index=X_step1.index,
)
# İmputer uygulanır ve sonuç aynı sütun adlarıyla DataFrame'e dönüştürülür.


if X_imputed.isna().any().any():
    # Eksik değer kalıp kalmadığı kontrol edilir.

    raise AssertionError(
        "Medyan doldurma sonrasında eksik descriptor kaldı."
    )
    # Eksik değer varsa pipeline durdurulur.


print("✅ CHECKPOINT 02.26: Eksik descriptor değerleri dolduruldu.")

## Hücre 27 — Sabit descriptorların çıkarılması

In [ ]:
constant_columns = [
    column
    for column in X_imputed.columns
    if X_imputed[column].nunique(
        dropna=False
    ) <= 1
]
# Bütün moleküllerde aynı değeri alan sabit descriptorlar belirlenir.


X_step2 = X_imputed.drop(
    columns=constant_columns,
).copy()
# Sabit descriptorlar feature matrisinden çıkarılır.


if X_step2.empty:
    # Sabit feature filtresi sonrasında descriptor kalıp kalmadığı kontrol edilir.

    raise ValueError(
        "Sabit feature filtresinden sonra descriptor kalmadı."
    )
    # Boş feature matrisiyle devam edilmez.


print("Sabit olduğu için çıkarılan:", len(constant_columns))
print("Kalan descriptor:", X_step2.shape[1])
print("✅ CHECKPOINT 02.27: Sabit descriptor filtresi tamamlandı.")

## Hücre 28 — Descriptor korelasyon matrisinin hesaplanması

In [ ]:
correlation_matrix = X_step2.corr(
    method="pearson",
).abs()
# Descriptorlar arasındaki mutlak Pearson korelasyon matrisi hesaplanır.


upper_triangle = correlation_matrix.where(
    np.triu(
        np.ones(correlation_matrix.shape),
        k=1,
    ).astype(bool)
)
# Her descriptor çiftini yalnızca bir kez incelemek için üst üçgen seçilir.


print("Korelasyon matrisi:", correlation_matrix.shape)
print("✅ CHECKPOINT 02.28: Korelasyon matrisi hesaplandı.")

## Hücre 29 — Yüksek korelasyonlu descriptorların çıkarılması

In [ ]:
high_correlation_columns = [
    column
    for column in upper_triangle.columns
    if (
        upper_triangle[column]
        > CORRELATION_THRESHOLD
    ).any()
]
# Başka bir descriptorla eşik üzeri korelasyon gösteren sütunlar belirlenir.


X_final = X_step2.drop(
    columns=high_correlation_columns,
).copy()
# Yüksek korelasyonlu descriptor çiftlerinden biri çıkarılır.


if X_final.empty:
    # Korelasyon filtresi sonrasında descriptor kalıp kalmadığı kontrol edilir.

    raise ValueError(
        "Korelasyon filtresinden sonra descriptor kalmadı."
    )
    # Boş feature matrisiyle devam edilmez.


if (
    X_final.isna().any().any()
    or not np.isfinite(
        X_final.to_numpy()
    ).all()
):
    # Final feature matrisinde eksik veya sonsuz değer olup olmadığı kontrol edilir.

    raise AssertionError(
        "Final feature matrisinde geçersiz değer kaldı."
    )
    # Geçersiz feature matrisi kaydedilmez.


print(
    "Yüksek korelasyon nedeniyle çıkarılan:",
    len(high_correlation_columns),
)
print("Final descriptor sayısı:", X_final.shape[1])
print("✅ CHECKPOINT 02.29: Korelasyon filtresi tamamlandı.")

## Hücre 30 — Model-ready feature store oluşturulması

In [ ]:
metadata_columns = [
    "molecule_id",
    "canonical_smiles",
    "target",
    "target_mean",
    "target_std",
    "source_rows",
]
# Modelleme dosyasının başında bulunacak metadata sütunları tanımlanır.


metadata_columns = [
    column
    for column in metadata_columns
    if column in df_clean.columns
]
# Gerçekten bulunan metadata sütunları seçilir.


feature_store = pd.concat(
    [
        df_clean[
            metadata_columns
        ].reset_index(drop=True),
        X_final.reset_index(drop=True),
    ],
    axis=1,
)
# Metadata, target ve final Mordred descriptorları yatay olarak birleştirilir.


feature_store.insert(
    2,
    "target_source_column",
    target_column,
)
# Targetın ham CSV'deki kaynak sütun adı saklanır.


feature_store.insert(
    3,
    "target_chembl_id",
    TARGET_ID,
)
# Her satıra ChEMBL Target ID eklenir.


if len(feature_store) != len(df_clean):
    # Feature store satır sayısı molekül sayısıyla karşılaştırılır.

    raise AssertionError(
        "Feature store satır sayısı molekül sayısıyla uyuşmuyor."
    )
    # Satır sayısı uyuşmazsa pipeline durdurulur.


display(feature_store.head(10))
print("Feature store boyutu:", feature_store.shape)
print("✅ CHECKPOINT 02.30: Model-ready feature store oluşturuldu.")

## Hücre 31 — Google Drive çıktı adlarının hazırlanması

In [ ]:
feature_filename = (
    f"{TARGET_ID}_Mordred2D_model_ready.csv"
)
# Sonraki notebookun GitHub'dan okuyacağı model-ready CSV adı oluşturulur.


raw_filename = (
    f"{TARGET_ID}_Mordred2D_raw_numeric.csv"
)
# Ham sayısal Mordred descriptor CSV adı oluşturulur.


manifest_filename = (
    f"{TARGET_ID}_Mordred2D_manifest.json"
)
# Feature manifest JSON dosya adı oluşturulur.


qc_filename = (
    f"{TARGET_ID}_Mordred2D_QC_summary.csv"
)
# Feature kalite kontrol CSV dosya adı oluşturulur.


invalid_filename = (
    f"{TARGET_ID}_Mordred2D_invalid_rows.csv"
)
# Geçersiz satır raporu dosya adı oluşturulur.


feature_path = DRIVE_ROOT / feature_filename
# Model-ready feature CSV Drive yolu oluşturulur.


raw_path = DRIVE_ROOT / raw_filename
# Ham descriptor CSV Drive yolu oluşturulur.


manifest_path = DRIVE_ROOT / manifest_filename
# Feature manifest Drive yolu oluşturulur.


qc_path = DRIVE_ROOT / qc_filename
# Kalite kontrol CSV Drive yolu oluşturulur.


invalid_path = DRIVE_ROOT / invalid_filename
# Geçersiz satır raporu Drive yolu oluşturulur.


print("✅ CHECKPOINT 02.31: Drive çıktı yolları hazırlandı.")

## Hücre 32 — CSV çıktılarının Google Drive'a kaydedilmesi

In [ ]:
feature_store.to_csv(
    feature_path,
    index=False,
)
# Model-ready Mordred feature store Google Drive'a kaydedilir.


mordred_numeric.to_csv(
    raw_path,
    index=False,
)
# Ham sayısal Mordred descriptor matrisi Google Drive'a kaydedilir.


invalid_rows.to_csv(
    invalid_path,
    index=False,
)
# Geçersiz target veya SMILES kayıtları Google Drive'a kaydedilir.


print("✅ CHECKPOINT 02.32: CSV çıktıları Drive'a kaydedildi.")

## Hücre 33 — Feature manifest dosyasının hazırlanması

In [ ]:
manifest = {
    "target_id": TARGET_ID,
    "github_input_path": USED_GITHUB_PATH,
    "github_input_url": USED_GITHUB_URL,
    "input_rows": int(len(df_raw)),
    "valid_unique_molecules": int(len(df_clean)),
    "invalid_rows": int(len(invalid_rows)),
    "target_column": target_column,
    "smiles_column": smiles_column,
    "id_column": id_column,
    "raw_descriptors": int(mordred_numeric.shape[1]),
    "max_missing_ratio": float(MAX_MISSING_RATIO),
    "correlation_threshold": float(CORRELATION_THRESHOLD),
    "dropped_high_missing": high_missing_columns,
    "dropped_constant": constant_columns,
    "dropped_high_correlation": high_correlation_columns,
    "final_feature_names": X_final.columns.tolist(),
    "final_feature_count": int(X_final.shape[1]),
}
# Feature üretimi, veri kaynağı ve temizlik bilgileri manifest sözlüğünde toplanır.


with open(
    manifest_path,
    "w",
    encoding="utf-8",
) as file:
    # Manifest dosyası yazma modunda açılır.

    json.dump(
        manifest,
        file,
        ensure_ascii=False,
        indent=2,
    )
    # Manifest okunabilir JSON biçiminde Google Drive'a yazılır.


print("✅ CHECKPOINT 02.33: Feature manifest Drive'a kaydedildi.")

## Hücre 34 — Feature kalite kontrol tablosunun hazırlanması

In [ ]:
qc_summary = pd.DataFrame(
    {
        "metric": [
            "input_rows",
            "valid_unique_molecules",
            "invalid_rows",
            "raw_descriptors",
            "dropped_high_missing",
            "dropped_constant",
            "dropped_high_correlation",
            "final_features",
            "final_missing_cells",
        ],
        "value": [
            len(df_raw),
            len(df_clean),
            len(invalid_rows),
            mordred_numeric.shape[1],
            len(high_missing_columns),
            len(constant_columns),
            len(high_correlation_columns),
            X_final.shape[1],
            int(X_final.isna().sum().sum()),
        ],
    }
)
# Feature üretim ve temizlik adımlarını özetleyen kalite kontrol tablosu oluşturulur.


display(qc_summary)
# Kalite kontrol tablosu notebook içinde gösterilir.


qc_summary.to_csv(
    qc_path,
    index=False,
)
# Kalite kontrol özeti Google Drive'a kaydedilir.


print("✅ CHECKPOINT 02.34: Feature kalite kontrol tablosu kaydedildi.")

## Hücre 35 — Drive çıktılarının son kontrolü

In [ ]:
OUTPUT_PATHS = [
    feature_path,
    raw_path,
    manifest_path,
    qc_path,
    invalid_path,
    target_plot_path,
]
# Üretilmesi beklenen bütün Drive çıktı yolları listelenir.


for output_path in OUTPUT_PATHS:
    # Bütün çıktılar sırayla kontrol edilir.

    if not output_path.exists():
        # Dosyanın gerçekten oluşup oluşmadığı kontrol edilir.

        raise IOError(
            f"Drive çıktısı oluşturulamadı: {output_path}"
        )
        # Eksik dosya varsa pipeline durdurulur.

    if output_path.stat().st_size == 0:
        # Dosyanın boş olup olmadığı kontrol edilir.

        raise IOError(
            f"Drive çıktısı boş oluşturuldu: {output_path}"
        )
        # Boş çıktı kabul edilmez.

    print(
        "[Drive kaydı]",
        output_path,
        f"({output_path.stat().st_size:,} byte)",
    )
    # Başarılı dosya yolu ve boyutu yazdırılır.


print()
print(
    "GitHub data/ klasörüne yükleyiniz:",
    feature_filename,
)
print(
    "✅ CHECKPOINT 02.35: Notebook tamamlandı ve "
    "bütün çıktılar Google Drive'a kaydedildi."
)

## Notebook sonu

Başarılı çalıştırma sonunda Google Drive içinde aşağıdaki ana dosya oluşur:

```text
CHEMBL206_Mordred2D_model_ready.csv
```

Bu dosyayı GitHub deposuna yükledikten sonra sonraki modelleme notebooku GitHub RAW bağlantısından okuyabilir.